# 01 RNA-seq のデータを読む

このノートブックでは、RNA-seq の解析で最初に手にする**カウント表**を開き、表の読み方を確かめます。

## この実習で使うデータ

ヒトの乳がん由来の培養細胞 MCF7 を使った実験のデータです。Sato らの論文（PNAS, 2024）で使われ、DDBJ に DRA011729 として公開されています。

- **shPPP3CA**：*PPP3CA* 遺伝子の働きを shRNA で弱めた細胞。*PPP3CA* は、脱リン酸化酵素カルシニューリンの本体（触媒サブユニット）の遺伝子です
- **shControl**：対照の shRNA を入れた細胞

どちらも、ドキシサイクリンを加えて2日間培養し、shRNA を働かせています。各条件3試料ずつ、合わせて6試料です。シーケンサーは NextSeq 500 で、1本のリードの長さは76塩基（single-end）です。

論文は、カルシニューリンが転写因子 E2F1 のタンパク質を安定にする、と報告しています。この実習では、*PPP3CA* を減らした細胞で、どの遺伝子の RNA 量が変わるかを調べます。

## カウント表ができるまで

デモで見たとおり、リードを参照ゲノム（GRCh38）に STAR で並べ、RSEM で遺伝子ごとの数にしました。遺伝子の情報には GENCODE v37 のタンパク質をコードする遺伝子を使っています。このノートブックから先は、できあがったカウント表を使います。

## 5つのノートブックの流れ

02 から先は、各ノートブックの最後に結果を zip でダウンロードし、次のノートブックの最初にその zip をアップロードして続きを進めます。

| ノートブック | 内容 | 最初に読み込むもの | 最後にダウンロードするもの |
|---|---|---|---|
| 01_Introduction | データと表の読み方 | `counts.csv`, `samples.csv`（配布元から） | なし |
| 02_QC_PCA | 試料の品質の確認と、試料どうしの似かた | `counts.csv`, `samples.csv`（配布元から） | `results_02.zip` |
| 03_DESeq2 | 2つの条件の比較 | `results_02.zip` | `results_03.zip` |
| 04_Visualization | 結果を図にする | `results_03.zip` | `results_04.zip` |
| 05_GSEA | 遺伝子のまとまりで見る | `results_04.zip` | `results_05.zip` |

## Colab の使い方

- 灰色の枠（セル）を選び、左の ▶ を押すか、Shift + Enter で実行します
- 上から順に実行してください。途中から始めると、前のセルで作った変数がなくてエラーになります
- 最初のセルを実行したときに「警告: このノートブックは Google が作成したものではありません」と出たら、「このまま実行」を押します
- エラーが出たら、まず「準備」のセルからもう一度実行してみてください
- このページを閉じると、Colab の中で作ったファイルは消えます。残したい結果は、最後のセルでダウンロードしてください

## 準備

下のセルを実行して、配布元から `counts.csv`, `samples.csv` をダウンロードします。

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # 計算には影響しない警告を表示しない

DATA_URL = "https://pub-fcc6b90fb32448a4b3b455e664df76f0.r2.dev/e2f1-rnaseq/v1/"
!wget -q -nc {DATA_URL}counts.csv
!wget -q -nc {DATA_URL}samples.csv
!ls -l counts.csv samples.csv

## 試料の情報を読む

`samples.csv` には、試料の名前と条件の対応が書いてあります。`accession` は、DDBJ に登録されたデータの番号です。

In [ ]:
import pandas as pd

samples = pd.read_csv("samples.csv", index_col="sample")
samples

## カウント表を読む

カウント表は、1行が1つの遺伝子、1列が1つの試料です。1つのマスには、その遺伝子に割り当てられたリードの数が入っています。

In [ ]:
counts = pd.read_csv("counts.csv", index_col="gene")
print("遺伝子の数:", counts.shape[0])
print("試料の数:", counts.shape[1])
counts.head()

### 1つのマスを読む

`counts.loc[遺伝子名, 試料名]` で、1つのマスの値を取り出せます。shControl_1 の *PPP3CA* のカウントを見てみます。

In [ ]:
counts.loc["PPP3CA", "shControl_1"]

### 1つの遺伝子を6試料で見比べる

下の `GENE` を書き換えて実行すると、ほかの遺伝子も見られます。遺伝子名は大文字で、`"` で囲みます。

- `"PPP3CA"`：shRNA で減らした遺伝子
- `"E2F1"`：論文で注目している転写因子
- `"GAPDH"`、`"ACTB"`：どの細胞でもよく使われている遺伝子

In [ ]:
GENE = "PPP3CA"

counts.loc[GENE]

## 試料ごとのカウントの合計

試料ごとに、全遺伝子のカウントを足してみます。

In [ ]:
total = counts.sum()
total

In [ ]:
import matplotlib.pyplot as plt

colors = ["#2F6FC4" if c == "shControl" else "#D9731A" for c in samples.loc[total.index, "condition"]]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(total.index, total / 1e6, color=colors)
ax.set_ylabel("Total counts (millions)")
ax.set_title("Total counts per sample")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

合計は試料ごとに少しずつ違います。合計が多い試料では、どの遺伝子のカウントも多めに出やすくなります。そのため、カウントをそのまま比べず、03 で合計などの違いを調整（正規化）してから比べます。

## どの試料でも0の遺伝子

6試料すべてでカウントが0の遺伝子を数えます。この細胞では使われていない遺伝子や、測れなかった遺伝子です。

In [ ]:
zero_in_all = (counts == 0).all(axis=1)
print("6試料すべてで0の遺伝子:", zero_in_all.sum())

## 確かめよう

1. shPPP3CA_2 の *E2F1* のカウントはいくつですか。
2. *PPP3CA* のカウントは、shControl と shPPP3CA でどう違いますか。shRNA で *PPP3CA* の RNA は減っていそうですか。
3. カウントの合計が最も多い試料と、最も少ない試料はどれですか。

次は **02_QC_PCA** に進みます。